# 特征提取

DeepDRiD regular-fundus-training，4 个模型。输出 `features/<model>.npz`（`image_ids`、`features`，模型原始输出）。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from fmspace.data import load_metadata
from fmspace.extract import extract, load_features, save_features
from fmspace.models import MODELS, load_encoder

DATA_DIR = Path(r"D:\path\to\DeepDRiD")  # 含 regular-fundus-training.csv 的目录
FEATURE_DIR = Path("../features")
DEVICE = None  # None：有 GPU 用 GPU

In [ ]:
metadata = load_metadata(DATA_DIR)
print(len(metadata), "张，", metadata["patient_id"].nunique(), "名患者")
print(metadata["eye_dr_level"].value_counts().sort_index().to_dict())
metadata.head()

## 试跑：前 8 张

In [ ]:
MODEL = "siglip"

encode = load_encoder(MODEL, DEVICE)
X = extract(encode, metadata["path"][:8], batch_size=4)
del encode
print(X.shape, X.dtype)
print("范数：", np.linalg.norm(X, axis=1).round(3))

## 全量提取（已存在的文件跳过）

In [ ]:
for name in MODELS:
    out = FEATURE_DIR / f"{name}.npz"
    if out.exists():
        print("跳过", out)
        continue
    X = extract(load_encoder(name, DEVICE), metadata["path"])
    save_features(out, metadata["image_id"], X)
    print(name, X.shape)

## 和旧特征比较

两边都 L2 归一化后，逐张算余弦相似度，应接近 1。汇总写入 `results/feature_check.csv`。

In [ ]:
OLD_DIR = Path(r"D:\path\to\SAMIT\features\DeepDRiD")


def l2(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)


rows = []
for name in MODELS:
    new = l2(load_features(FEATURE_DIR / f"{name}.npz", metadata["image_id"]))
    old = l2(load_features(OLD_DIR / f"{name}.npz", metadata["image_id"]))
    cos = (new * old).sum(axis=1)
    rows.append({"model": name, "n": len(cos), "cos_min": cos.min(), "cos_median": np.median(cos)})

check = pd.DataFrame(rows)
Path("../results").mkdir(exist_ok=True)
check.to_csv("../results/feature_check.csv", index=False)
check